In [1]:
import os
import random
import pandas as pd
import shutil
import git
import fnmatch
from tqdm import tqdm

# Change it to your google drive path where this notebook located.
drive_path = '/Users/samyiin/Projects/ZipfLawAnalysis'
os.chdir(drive_path)

from Utils.PythonParser import PythonIdentifierExtractor

# I don't need these warnings
import warnings


In [2]:
df_users = pd.read_csv('Database/TempData/GatherData/Filter_4_SendUserEmail/Results/all_users.csv')

In [3]:
def extract_identifeir_single_user(python_identifier_extractor, dir_cleaned_python_files_dir):
    list_df_names = []
    # walk through the directory
    for foldername, subfolders, filenames in os.walk(dir_cleaned_python_files_dir):
        for filename in filenames:
            if fnmatch.fnmatch(filename, '*.py'):
                python_file_path = os.path.join(foldername, filename)
                try:
                    # don't print error messages
                    df_names = python_identifier_extractor.extract_identifiers_without_cleaning(python_file_path)
                except:
                    continue
                list_df_names.append(df_names)

    # see if this user failed
    if len(list_df_names) ==0:
        return 
    df_names = pd.concat(list_df_names, ignore_index=True)
    df_names.to_csv(os.path.join(dir_cleaned_python_files_dir, "df_names.csv"), index=False)
    
def extract_identifeir_all_users(df_users):
    python_identifier_extractor = PythonIdentifierExtractor()
    for i in tqdm(range(len(df_users))):
        user_login = df_users.iloc[i].to_dict()['login']
        # go to the directory for this user
        user_directory_path = os.path.join('Database/UserData', str(user_login))
        dir_cleaned_python_files_dir = os.path.join(user_directory_path, 'CleanedPythonFiles')
        
        # don't need to cache the result in this operation, this process is pretty fast....
        
        # parse all this user's python files, we saved them under Database/UserData/<user_login>/CleanedPythonFiles/
        with warnings.catch_warnings(): # ignore the syntax error warnings 
            warnings.simplefilter("ignore", SyntaxWarning)
            extract_identifeir_single_user(python_identifier_extractor, dir_cleaned_python_files_dir)
    
        

extract_identifeir_all_users(df_users)

100%|█████████████████████████████████████████████████████████████████████████████████████████| 576/576 [00:11<00:00, 48.49it/s]


# Gather all the names we got

In [4]:
def gather_all_names(df_users):
    list_df_names = []
    for i in tqdm(range(len(df_users))):
        user_login = df_users.iloc[i].to_dict()['login']
        # go to the directory for this user
        user_directory_path = os.path.join('Database/UserData', str(user_login))
        dir_save_python_file = os.path.join(user_directory_path, 'CleanedPythonFiles')
        df_names_csv_file_path = os.path.join(dir_save_python_file, "df_names.csv")
        # open the csv files
        if os.path.exists(df_names_csv_file_path):
            try:
                df_names = pd.read_csv(df_names_csv_file_path)
                df_names["login"] = user_login
                # df_names["country"] = df_users.iloc[i].to_dict()['country']
                list_df_names.append(df_names)
            except:
                # we parsed them, but there are no names at all(usually single empty python file)
                print(f"There is something wrong with hidden user_login")    
        else:
            print(f"User hidden user_login does not have df_names")
        
    df_names = pd.concat(list_df_names, ignore_index=True)
    df_names.to_csv("Database/TempData/DataProcessing/df_hardwords.csv", index=False)
    return df_names
df_names = gather_all_names(df_users)

 40%|██████████████████████████████████▉                                                    | 231/576 [00:00<00:00, 1120.80it/s]

User hidden user_login does not have df_names
User hidden user_login does not have df_names
User hidden user_login does not have df_names
User hidden user_login does not have df_names
There is something wrong with hidden user_login
User hidden user_login does not have df_names


 80%|█████████████████████████████████████████████████████████████████████▋                 | 461/576 [00:00<00:00, 1078.15it/s]

There is something wrong with hidden user_login
There is something wrong with hidden user_login
User hidden user_login does not have df_names


100%|███████████████████████████████████████████████████████████████████████████████████████| 576/576 [00:00<00:00, 1029.86it/s]
